# Baseline: Обучение Multi-Branch MLP на размеченных данных


In [14]:
import os
import sys
import warnings

import pytorch_lightning as pl

warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

from model import MultiBranchMLP
from data_module import DataModule
from lightning_module import BaseLightningModule

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping


def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    import random
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
torch.set_float32_matmul_precision('high')

## 1. Загрузка данных


In [15]:
data_dir = '../data'

dm = DataModule(
    data_dir=data_dir,
    batch_size=128,
    num_workers=4
)

dm.setup()

print(f'Input dimension: {dm.input_dim}')
print(f'Number of classes: {dm.n_classes}')
print(f'Labeled train samples: {len(dm.train_labeled_dataset)}')
print(f'Test samples: {len(dm.test_dataset)}')


Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000
Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000


## 2. Анализ дисбаланса классов и вычисление весов


In [16]:
train_labels = dm.train_labeled_dataset.y

unique_labels = np.unique(train_labels)
class_weights = compute_class_weight(
    'balanced',
    classes=unique_labels,
    y=train_labels
)

print(f'Class weights: {dict(zip(unique_labels, class_weights))}')

class_weights_tensor = torch.FloatTensor(class_weights)
class_weights = class_weights_tensor

Class weights: {np.int64(0): np.float64(1.0738255033557047), np.int64(1): np.float64(0.963855421686747), np.int64(2): np.float64(1.0256410256410255), np.int64(3): np.float64(1.103448275862069), np.int64(4): np.float64(0.9523809523809523), np.int64(5): np.float64(0.935672514619883), np.int64(6): np.float64(0.9523809523809523), np.int64(7): np.float64(1.0596026490066226), np.int64(8): np.float64(0.9248554913294798), np.int64(9): np.float64(1.0457516339869282)}


## 3. Создание модели


In [17]:
model = MultiBranchMLP(
    input_dim=dm.input_dim,
    hidden_dim=256,
    output_dim=dm.n_classes,
    num_blocks=4,
    dropout=0.1,
    combine_mode='concat'
)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')


Model parameters: 4,080,650


## 4. Создание Lightning модуля


In [18]:
checkpoint_callback = ModelCheckpoint(
    dirpath='checkpoints',
    filename='best_model-{epoch:02d}-{val_accuracy:.4f}',
    monitor='val_accuracy',
    mode='max',
    save_top_k=1,
    save_last=True
    )


early_stopping = EarlyStopping(
    monitor='val_f1_macro',
    patience=5,
    mode='max',
)

weight_tensor = torch.tensor(class_weights, dtype=torch.float32)
loss_fn = nn.CrossEntropyLoss(weight=weight_tensor)

def train_model(
    model,
    dm,
    max_epochs=10,
    lr=1e-3,
    optimizer_type='adam'
):
    lightning_model = BaseLightningModule(
        model=model,
        loss_fn=loss_fn,
        optimizer_type=optimizer_type,
        learning_rate=lr,
        task_type='multiclass'
    )


    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        callbacks=[early_stopping],
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False
    )
    trainer.fit(lightning_model, dm)

    metrics = trainer.callback_metrics
    return {
        'val_acc': metrics.get('val_accuracy', 0).item(),
        'val_f1': metrics.get('val_f1_macro', 0).item()
    }


## 5. Обучение модели


In [18]:
hidden_dims = [64,128,256,512]
depths = [2,4,6,8,10,12]
lrs = [1e-3,1e-2,1e-4]
optimizers = ['rmsprop',"adamw"]
combine_modes=['concat','sum']
dropouts = [0.1]
activations = ['relu','gelu']
best_val_f1 = 0.0
best_config = None
max_epochs = 100

for hd in hidden_dims:
    for nb in depths:
        for lr in lrs:
            for opt in optimizers:
                for combine_mode in combine_modes:
                    for dropout in dropouts:
                        for activation in activations:
                             model = MultiBranchMLP(
                                 input_dim=dm.input_dim,
                                 hidden_dim=hd,
                                 output_dim=dm.n_classes,
                                 num_blocks=nb,
                                 dropout=dropout,
                                 combine_mode=combine_mode,
                                 activation=activation
                             )

                             metrics = train_model(
                                 model,
                                 dm,
                                 max_epochs=max_epochs,
                                 lr=lr,
                                 optimizer_type=opt
                             )
                             val_f1 = metrics['val_f1']
                             val_acc = metrics['val_acc']
                             print(f'hidden_dim={hd}, num_blocks={nb}, lr={lr}, opt={opt} -> val_f1={val_f1:.4f}, val_acc={val_acc:.4f} dropout={dropout:.4f} best_combine_mode={combine_modes} activation={activation}')
                             if val_f1 > best_val_f1:
                                 best_val_f1 = val_f1
                                 best_config = (hd, nb, lr, opt,combine_mode,dropout,activation)

best_hidden_dim, best_depth, best_lr, best_optimizer,best_combine_mode,best_dropout,best_activation = best_config
print(f'Best configuration: hidden_dim={best_hidden_dim}, num_blocks={best_depth}, lr={best_lr}, opt={best_optimizer}, val_f1={best_val_f1:.4f}, best_combine_mode={best_combine_mode}  best_dropout={best_dropout} best_activation={best_activation}')

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000
Epoch 0: accuracy=0.0703, f1_macro=0.0212
Epoch 0: accuracy=0.1031, f1_macro=0.0250
Epoch 1: accuracy=0.1105, f1_macro=0.0480
Epoch 2: accuracy=0.1260, f1_macro=0.0723
Epoch 3: accuracy=0.1393, f1_macro=0.0949
Epoch 4: accuracy=0.1393, f1_macro=0.1062
Epoch 5: accuracy=0.1530, f1_macro=0.1305
Epoch 6: accuracy=0.1589, f1_macro=0.1490
Epoch 7: accuracy=0.1658, f1_macro=0.1565
Epoch 8: accuracy=0.1713, f1_macro=0.1602
Epoch 9: accuracy=0.1756, f1_macro=0.1646
Epoch 10: accuracy=0.1759, f1_macro=0.1659
Epoch 11: accuracy=0.1820, f1_macro=0.1697
Epoch 12: accuracy=0.1837, f1_macro=0.1745
Epoch 13: accuracy=0.1830, f1_macro=0.1755
Epoch 14: accuracy=0.1792, f1_macro=0.1716
Epoch 15: accuracy=0.1798, f1_macro=0.1701
Epoch 16: accuracy=0.1759, f1_macro=0.1662
Epoch 17: accuracy=0.1765, f1_macro=0.1658
Epoch 18: accuracy=0.1772, f1_macro=0.1672


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


hidden_dim=64, num_blocks=2, lr=0.001, opt=rmsprop -> val_f1=0.1672, val_acc=0.1772 dropout=0.1000 best_combine_mode=['concat', 'sum'] activation=relu


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000



Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

KeyboardInterrupt: 

## 6. Оценка на тестовой выборке


In [13]:
best_model_path = checkpoint_callback.best_model_path
print(f'Loading best model from: {best_model_path}')


best_model = BaseLightningModule.load_from_checkpoint(
    best_model_path,
    model=model,
    loss_fn=loss_fn,
    optimizer_type='adamw',
    learning_rate=1e-3,
    task_type='multiclass'
)
trainer = Trainer(
    max_epochs=max_epochs,
    callbacks=[checkpoint_callback,early_stopping],
    enable_checkpointing=True,
    logger=True,
    enable_progress_bar=True,
    enable_model_summary=True,
    accelerator='auto',
    devices='auto'
)

test_results = trainer.test(best_model, dm)

print('\n=== Финальные результаты на тестовой выборке ===')
for key, value in test_results[0].items():
    print(f'{key}: {value:.4f}')


Loading best model from: 


PermissionError: [Errno 13] Permission denied: 'C:/Users/ART PRONKIN/PycharmProjects/DeepMachineLearning/lesson7/homework/baseline'